# Notebook 83: HITL V8.9 vs A1 vs A3 Whole-Page Comparison

**Runtime:** CPU. Select **Runtime → Run all**.

## Purpose

Run the three relevant correction candidates on the actual reviewed HITL pages:

- **V89_INCUMBENT**: frozen V8.9 output, including Ring-2 restoration.
- **A1_PRESERVE_ALL_AMBIGUOUS**: V8.9, but every AMBIGUOUS pixel is restored to the observed byte.
- **A3_PIXEL_GATE**: V8.9, but only AMBIGUOUS pixels with `p_print >= 0.50` are restored to the observed byte.

The primary whole-page review is:

`OBSERVED | V8.9 | A1 | A3`

The key diagnostic is **A1 versus A3**. Those disagreement pixels are where A1 preserves the observed page but A3 returns to the V8.9/Telea treatment.

## Boundaries

- Uses reviewed corrected masks where an authoritative review audit exists.
- Does not use clean truth.
- Does not retrain, refit, recalibrate, or alter V8.9 or `p_print`.
- Does not authorise production promotion.
- Preserves pixels outside the prepared authoritative mask byte-for-byte.
- Records and skips zero-authority and zero-AMBIGUOUS pages.
- Is resumable. Complete page outputs are reused; partial page directories fail closed.

The design follows the project contract that annotation confidence and correction safety remain separate, intervention is limited to the authoritative mask, and reconstruction is restricted to the explicit AMBIGUOUS class.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy matplotlib pyarrow


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util, json, hashlib, shutil, gc, sys, platform
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')

# Frozen V8.9 runtime objects carried forward from Notebook 82.
V89_OPERATOR_PATH = ROOT/'v8_9_release'/'v8_9_operator.py'
P_PRINT_BUNDLE = ROOT/'observed_print_full_development_freeze_v8_2f'/'20261117_120424_v8_2f_full_development_p_print_freeze'/'p_print_provider'
REMOVE_BUNDLE = ROOT/'m5_remove_full_development_freeze_v8_8f'/'20261117_110512_notebook52f_v8_8_full_development_remove_freeze'/'full_development_remove'

# Authoritative HITL lineage.
SOURCE_DIR = ROOT/'source'/'annotated_only'
REVIEW_AUDIT_DIR = ROOT/'human_in_the_middle'/'audit'
DRAFT_AUDIT_DIR = ROOT/'human_in_the_middle'/'draft_audit'

OUTPUT_ROOT = ROOT/'notebook83_hitl_v89_a1_a3_comparison'
RUN_OVERRIDE = None
PPRINT_GATE = 0.50
EXPECTED_REVIEWED_PAGES = 151
MAX_CONTACT_PAGES = 20
CROP_MARGIN = 90
SUPPORTED = {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

if RUN_OVERRIDE is None:
    RUN = OUTPUT_ROOT/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook83')
    for p in [RUN,RUN/'pages',RUN/'tables',RUN/'review',RUN/'audit',RUN/'upload']:
        p.mkdir(parents=True,exist_ok=False)
else:
    RUN = Path(RUN_OVERRIDE)
    for p in [RUN,RUN/'pages',RUN/'tables',RUN/'review',RUN/'audit',RUN/'upload']:
        p.mkdir(parents=True,exist_ok=True)

def require(value,message):
    if not bool(value): raise RuntimeError(message)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def md5_file(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256()
    h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes())
    return h.hexdigest()

def read_gray(path):
    im=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)
    require(im is not None,f'Unreadable image: {path}')
    return im

def image_files(folder):
    folder=Path(folder)
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED) if folder.exists() else []

print('Run:',RUN)


## 1. Load and verify the actual frozen V8.9 operator

Notebook 83 derives all candidates from the operator output. It does not approximate V8.9 from an earlier implementation.


In [ ]:
require(V89_OPERATOR_PATH.exists(),f'Missing V8.9 operator: {V89_OPERATOR_PATH}')
spec=importlib.util.spec_from_file_location('v8_9_operator',V89_OPERATOR_PATH)
mod=importlib.util.module_from_spec(spec)
sys.modules['v8_9_operator']=mod
spec.loader.exec_module(mod)
op=mod.load_v89_operator(P_PRINT_BUNDLE,REMOVE_BUNDLE)
require(type(op).__name__=='V89Operator',f'Expected V89Operator, got {type(op).__name__}')
print('V8.9 operator SHA-256:',sha256_file(V89_OPERATOR_PATH))


## 2. Resolve the 151 reviewed HITL pages by audit identity

The review audit is authoritative. Each source is matched by source MD5, then the audit's `aligned_output` and `corrected_mask_output` are used. Draft masks are not substituted for reviewed masks.


In [ ]:
# ============================================================
# 2. Resolve COMPLETE reviewed HITL pages by audit identity
#
# Incomplete / half-built HITL packages are valid exclusions
# from this experiment.
#
# Required for inclusion:
#   - matching review audit
#   - aligned_output present in audit
#   - corrected_mask_output present in audit
#   - both referenced files exist
#   - both images are readable
#   - observed and mask shapes agree
#
# Anything else is recorded and SKIPPED.
# ============================================================


def read_audits(
    folder,
    label,
):
    records = []
    failures = []

    for path in sorted(
        Path(folder).glob("*.json")
    ):

        try:

            value = json.loads(
                path.read_text()
            )

            require(
                isinstance(
                    value,
                    dict,
                ),
                (
                    f"{path}: audit root "
                    "is not an object"
                ),
            )

            value[
                "_audit_path"
            ] = str(path)

            value[
                "_audit_type"
            ] = label

            records.append(
                value
            )

        except Exception as e:

            failures.append({
                "audit_type":
                    label,

                "audit_path":
                    str(path),

                "error":
                    str(e),
            })

    return (
        records,
        failures,
    )


reviews, review_failures = read_audits(
    REVIEW_AUDIT_DIR,
    "review",
)

drafts, draft_failures = read_audits(
    DRAFT_AUDIT_DIR,
    "draft",
)


# ------------------------------------------------------------
# A malformed JSON audit is fundamentally different from an
# incomplete review package.
#
# Malformed audit records remain a hard failure because their
# contents cannot be interpreted reliably.
# ------------------------------------------------------------

audit_parse_failures = (
    review_failures
    + draft_failures
)

if audit_parse_failures:

    pd.DataFrame(
        audit_parse_failures
    ).to_csv(
        RUN
        / "audit"
        / "audit_parse_failures.csv",
        index=False,
    )

    raise RuntimeError(
        f"{len(audit_parse_failures)} audit file(s) "
        "could not be parsed. "
        "See audit_parse_failures.csv."
    )


# ------------------------------------------------------------
# Index review audits by source MD5.
# ------------------------------------------------------------

review_by_md5 = {}

duplicate_review_rows = []


for review in reviews:

    source_md5 = str(
        review.get(
            "source_md5",
            "",
        )
    ).lower()

    if not source_md5:
        continue

    if source_md5 in review_by_md5:

        duplicate_review_rows.append({
            "source_md5":
                source_md5,

            "first_audit":
                review_by_md5[
                    source_md5
                ][
                    "_audit_path"
                ],

            "second_audit":
                review[
                    "_audit_path"
                ],
        })

        continue

    review_by_md5[
        source_md5
    ] = review


if duplicate_review_rows:

    pd.DataFrame(
        duplicate_review_rows
    ).to_csv(
        RUN
        / "audit"
        / "duplicate_review_audits.csv",
        index=False,
    )

    raise RuntimeError(
        "Duplicate review audits found. "
        "See duplicate_review_audits.csv."
    )


# ============================================================
# Build INCLUDED roster and explicit EXCLUSION table
# ============================================================

roster_rows = []
exclusion_rows = []


for source in image_files(
    SOURCE_DIR
):

    source_md5 = (
        md5_file(source)
        .lower()
    )

    review = review_by_md5.get(
        source_md5
    )


    # --------------------------------------------------------
    # No completed review audit
    # --------------------------------------------------------

    if review is None:

        exclusion_rows.append({
            "page_key":
                source.stem,

            "source_file":
                str(source),

            "source_md5":
                source_md5,

            "status":
                "SKIPPED_NO_REVIEW_AUDIT",

            "reason":
                (
                    "No review audit matches "
                    "the source MD5."
                ),

            "review_audit_path":
                "",
        })

        print(
            f"{source.stem}: SKIP - "
            "no review audit"
        )

        continue


    audit_path = str(
        review.get(
            "_audit_path",
            "",
        )
    )


    # --------------------------------------------------------
    # The audit exists but aligned_output was never written.
    #
    # This is the Page137 situation.
    # --------------------------------------------------------

    aligned_raw = review.get(
        "aligned_output"
    )

    if not aligned_raw:

        exclusion_rows.append({
            "page_key":
                source.stem,

            "source_file":
                str(source),

            "source_md5":
                source_md5,

            "status":
                "SKIPPED_INCOMPLETE_REVIEW_PACKAGE",

            "reason":
                "review_audit_missing_aligned_output",

            "review_audit_path":
                audit_path,
        })

        print(
            f"{source.stem}: SKIP - "
            "review audit exists but "
            "aligned_output is missing"
        )

        continue


    # --------------------------------------------------------
    # The audit exists but corrected_mask_output was never
    # written.
    # --------------------------------------------------------

    mask_raw = review.get(
        "corrected_mask_output"
    )

    if not mask_raw:

        exclusion_rows.append({
            "page_key":
                source.stem,

            "source_file":
                str(source),

            "source_md5":
                source_md5,

            "status":
                "SKIPPED_INCOMPLETE_REVIEW_PACKAGE",

            "reason":
                (
                    "review_audit_missing_"
                    "corrected_mask_output"
                ),

            "review_audit_path":
                audit_path,
        })

        print(
            f"{source.stem}: SKIP - "
            "review audit exists but "
            "corrected_mask_output is missing"
        )

        continue


    # --------------------------------------------------------
    # Resolve referenced files.
    # --------------------------------------------------------

    observed_path = Path(
        aligned_raw
    )

    seed_path = Path(
        mask_raw
    )


    if not observed_path.exists():

        exclusion_rows.append({
            "page_key":
                source.stem,

            "source_file":
                str(source),

            "source_md5":
                source_md5,

            "status":
                "SKIPPED_INCOMPLETE_REVIEW_PACKAGE",

            "reason":
                (
                    "aligned_output_file_missing"
                ),

            "review_audit_path":
                audit_path,

            "expected_path":
                str(
                    observed_path
                ),
        })

        print(
            f"{source.stem}: SKIP - "
            "aligned_output file does not exist"
        )

        continue


    if not seed_path.exists():

        exclusion_rows.append({
            "page_key":
                source.stem,

            "source_file":
                str(source),

            "source_md5":
                source_md5,

            "status":
                "SKIPPED_INCOMPLETE_REVIEW_PACKAGE",

            "reason":
                (
                    "corrected_mask_output_"
                    "file_missing"
                ),

            "review_audit_path":
                audit_path,

            "expected_path":
                str(
                    seed_path
                ),
        })

        print(
            f"{source.stem}: SKIP - "
            "corrected mask file does not exist"
        )

        continue


    # --------------------------------------------------------
    # Images must also actually be readable.
    # --------------------------------------------------------

    observed = cv2.imread(
        str(
            observed_path
        ),
        cv2.IMREAD_GRAYSCALE,
    )

    seed = cv2.imread(
        str(
            seed_path
        ),
        cv2.IMREAD_GRAYSCALE,
    )


    if observed is None:

        exclusion_rows.append({
            "page_key":
                source.stem,

            "source_file":
                str(source),

            "source_md5":
                source_md5,

            "status":
                "SKIPPED_INVALID_REVIEW_PACKAGE",

            "reason":
                "aligned_output_unreadable",

            "review_audit_path":
                audit_path,

            "path":
                str(
                    observed_path
                ),
        })

        print(
            f"{source.stem}: SKIP - "
            "aligned_output unreadable"
        )

        continue


    if seed is None:

        exclusion_rows.append({
            "page_key":
                source.stem,

            "source_file":
                str(source),

            "source_md5":
                source_md5,

            "status":
                "SKIPPED_INVALID_REVIEW_PACKAGE",

            "reason":
                "corrected_mask_output_unreadable",

            "review_audit_path":
                audit_path,

            "path":
                str(
                    seed_path
                ),
        })

        print(
            f"{source.stem}: SKIP - "
            "corrected mask unreadable"
        )

        continue


    # --------------------------------------------------------
    # Shape agreement is required for an authoritative
    # intervention mask.
    # --------------------------------------------------------

    if observed.shape != seed.shape:

        exclusion_rows.append({
            "page_key":
                source.stem,

            "source_file":
                str(source),

            "source_md5":
                source_md5,

            "status":
                "SKIPPED_INVALID_REVIEW_PACKAGE",

            "reason":
                "observed_mask_shape_mismatch",

            "review_audit_path":
                audit_path,

            "observed_shape":
                str(
                    observed.shape
                ),

            "mask_shape":
                str(
                    seed.shape
                ),
        })

        print(
            f"{source.stem}: SKIP - "
            f"shape mismatch "
            f"{observed.shape} vs {seed.shape}"
        )

        continue


    # ========================================================
    # Complete reviewed HITL page
    # ========================================================

    roster_rows.append({
        "page_key":
            source.stem,

        "source_file":
            str(source),

        "source_md5":
            source_md5,

        "observed_path":
            str(
                observed_path
            ),

        "seed_path":
            str(
                seed_path
            ),

        "review_audit_path":
            audit_path,

        "mask_status":
            "reviewed_corrected",

        "height":
            int(
                observed.shape[0]
            ),

        "width":
            int(
                observed.shape[1]
            ),

        "seed_pixels":
            int(
                (
                    seed > 0
                ).sum()
            ),

        "status":
            "INCLUDED",
    })


# ============================================================
# Save complete roster and exclusions
# ============================================================

roster = pd.DataFrame(
    roster_rows
)

exclusions = pd.DataFrame(
    exclusion_rows
)


require(
    len(roster) > 0,
    (
        "No complete reviewed HITL pages "
        "remain after package validation."
    ),
)


roster = (
    roster
    .sort_values(
        "page_key"
    )
    .reset_index(
        drop=True
    )
)


roster.to_csv(
    RUN
    / "tables"
    / "hitl_reviewed_roster.csv",
    index=False,
)


exclusions.to_csv(
    RUN
    / "tables"
    / "hitl_input_exclusions.csv",
    index=False,
)


# ============================================================
# Report what actually exists
# ============================================================

print()
print("=" * 72)
print("HITL INPUT-RESOLUTION RESULT")
print("=" * 72)

print(
    "Complete reviewed packages:",
    len(roster),
)

print(
    "Excluded / incomplete packages:",
    len(exclusions),
)

print()


if len(exclusions):

    print(
        exclusions[
            [
                "page_key",
                "status",
                "reason",
            ]
        ].to_string(
            index=False
        )
    )


print()
print(
    "Experiment will run only on the "
    "complete reviewed packages above."
)

## 3. Candidate construction and resumability

Definitions:

- `V89_INCUMBENT`: exact operator output.
- `A1_PRESERVE_ALL_AMBIGUOUS`: start from V8.9 and replace all AMBIGUOUS bytes with observed bytes.
- `A3_PIXEL_GATE`: start from V8.9 and replace only `AMBIGUOUS & finite(p_print) & p_print >= 0.50` with observed bytes.

Because both challengers start from the actual V8.9 page, every V8.9 post-treatment, including Ring 2, remains identical except where the candidate explicitly restores an AMBIGUOUS observed byte.


In [ ]:
REQUIRED_PAGE_FILES=[
    'observed.png','seed.png','authority.png','ambiguous.png','ring2.png',
    'p_print.npy','V89_INCUMBENT.png','A1_PRESERVE_ALL_AMBIGUOUS.png',
    'A3_PIXEL_GATE.png','a1_vs_v89.png','a3_vs_v89.png','a1_vs_a3.png',
    'a3_gate.png','page_audit.json'
]

def complete_page_dir(d):
    return d.is_dir() and all((d/name).is_file() for name in REQUIRED_PAGE_FILES)

def candidate_arrays(observed,seed,page):
    result=op.run_page(observed,seed,page=page)
    v89=np.asarray(result.corrected).copy()
    authority=np.asarray(result.prepared_mask)>0 if hasattr(result,'prepared_mask') else mod.prepare_mask(seed,2)>0
    ambiguous=np.asarray(result.ambiguous_mask)>0 if hasattr(result,'ambiguous_mask') else np.asarray(result.ambiguous)>0
    ring2=np.asarray(result.ring2_mask)>0
    p_print=np.asarray(result.p_print,dtype=np.float32)
    require(v89.shape==observed.shape==authority.shape==ambiguous.shape==ring2.shape==p_print.shape,f'{page}: operator shape mismatch')
    gate=ambiguous & np.isfinite(p_print) & (p_print>=PPRINT_GATE)
    a1=v89.copy(); a1[ambiguous]=observed[ambiguous]
    a3=v89.copy(); a3[gate]=observed[gate]
    for name,img in [('V89',v89),('A1',a1),('A3',a3)]:
        require(np.array_equal(img[~authority],observed[~authority]),f'{page}: {name} outside-authority drift')
    require(np.array_equal(a1[~ambiguous],v89[~ambiguous]),f'{page}: A1 changed non-AMBIGUOUS pixels')
    require(np.array_equal(a3[~gate],v89[~gate]),f'{page}: A3 changed non-gated pixels')
    require(np.array_equal(a1[ambiguous],observed[ambiguous]),f'{page}: A1 did not preserve AMBIGUOUS exactly')
    require(np.array_equal(a3[gate],observed[gate]),f'{page}: A3 did not preserve gate exactly')
    return {'v89':v89,'a1':a1,'a3':a3,'authority':authority,'ambiguous':ambiguous,'ring2':ring2,'p_print':p_print,'gate':gate}


## 4. Run or resume all reviewed HITL pages

Zero-authority and zero-AMBIGUOUS pages are valid exclusions from the treatment comparison. They are recorded rather than treated as failures.


In [ ]:
rows=[]
for _,r in roster.iterrows():
    key=str(r.page_key); d=RUN/'pages'/key
    if complete_page_dir(d):
        rec=json.loads((d/'page_audit.json').read_text()); rec['execution_status']='REUSED'; rows.append(rec); print(key,'REUSE'); continue
    if d.exists():
        raise RuntimeError(f'{key}: partial page directory found at {d}. Remove only this directory after inspection, then resume.')
    observed=read_gray(r.observed_path); seed=read_gray(r.seed_path)
    if not (seed>0).any():
        rows.append({'page_key':key,'status':'SKIPPED_ZERO_AUTHORITY','execution_status':'SKIPPED','seed_pixels':0,'ambiguous_pixels':0})
        print(key,'SKIP zero authority'); continue
    try:
        z=candidate_arrays(observed,seed,key)
    except RuntimeError as e:
        msg=str(e).lower()
        if any(x in msg for x in ['no m3','m3 pixels','no ambiguous','ambiguous pixels']):
            rows.append({'page_key':key,'status':'SKIPPED_ZERO_AMBIGUOUS','execution_status':'SKIPPED','seed_pixels':int((seed>0).sum()),'ambiguous_pixels':0,'skip_reason':str(e)})
            print(key,'SKIP zero AMBIGUOUS'); gc.collect(); continue
        raise
    if not z['ambiguous'].any():
        rows.append({'page_key':key,'status':'SKIPPED_ZERO_AMBIGUOUS','execution_status':'SKIPPED','seed_pixels':int((seed>0).sum()),'authority_pixels':int(z['authority'].sum()),'ambiguous_pixels':0})
        print(key,'SKIP zero AMBIGUOUS'); del z; gc.collect(); continue
    d.mkdir(parents=True,exist_ok=False)
    cv2.imwrite(str(d/'observed.png'),observed); cv2.imwrite(str(d/'seed.png'),seed)
    cv2.imwrite(str(d/'authority.png'),z['authority'].astype(np.uint8)*255)
    cv2.imwrite(str(d/'ambiguous.png'),z['ambiguous'].astype(np.uint8)*255)
    cv2.imwrite(str(d/'ring2.png'),z['ring2'].astype(np.uint8)*255)
    np.save(d/'p_print.npy',z['p_print'])
    cv2.imwrite(str(d/'V89_INCUMBENT.png'),z['v89'])
    cv2.imwrite(str(d/'A1_PRESERVE_ALL_AMBIGUOUS.png'),z['a1'])
    cv2.imwrite(str(d/'A3_PIXEL_GATE.png'),z['a3'])
    diff_a1_v89=(z['a1']!=z['v89']); diff_a3_v89=(z['a3']!=z['v89']); diff_a1_a3=(z['a1']!=z['a3'])
    cv2.imwrite(str(d/'a1_vs_v89.png'),diff_a1_v89.astype(np.uint8)*255)
    cv2.imwrite(str(d/'a3_vs_v89.png'),diff_a3_v89.astype(np.uint8)*255)
    cv2.imwrite(str(d/'a1_vs_a3.png'),diff_a1_a3.astype(np.uint8)*255)
    cv2.imwrite(str(d/'a3_gate.png'),z['gate'].astype(np.uint8)*255)
    ambiguous_pixels=int(z['ambiguous'].sum()); gate_pixels=int(z['gate'].sum())
    rec={
        'page_key':key,'status':'PROCESSED','execution_status':'PROCESSED',
        'source_file':str(r.source_file),'observed_path':str(r.observed_path),'seed_path':str(r.seed_path),
        'seed_pixels':int((seed>0).sum()),'authority_pixels':int(z['authority'].sum()),
        'ambiguous_pixels':ambiguous_pixels,'ring2_pixels':int(z['ring2'].sum()),
        'a3_gate_pixels':gate_pixels,'a3_gate_fraction_of_ambiguous':gate_pixels/max(ambiguous_pixels,1),
        'a1_vs_v89_changed_pixels':int(diff_a1_v89.sum()),
        'a3_vs_v89_changed_pixels':int(diff_a3_v89.sum()),
        'a1_vs_a3_changed_pixels':int(diff_a1_a3.sum()),
        'a3_retelea_fraction_vs_a1':float(diff_a1_a3.sum()/max(ambiguous_pixels,1)),
        'p_print_ambiguous_mean':float(np.nanmean(z['p_print'][z['ambiguous']])),
        'p_print_ambiguous_median':float(np.nanmedian(z['p_print'][z['ambiguous']])),
        'outside_authority_drift_v89':int(np.sum(z['v89'][~z['authority']]!=observed[~z['authority']])),
        'outside_authority_drift_a1':int(np.sum(z['a1'][~z['authority']]!=observed[~z['authority']])),
        'outside_authority_drift_a3':int(np.sum(z['a3'][~z['authority']]!=observed[~z['authority']])),
    }
    (d/'page_audit.json').write_text(json.dumps(rec,indent=2))
    rows.append(rec); print(key,'PROCESSED','ambiguous',ambiguous_pixels,'A3 gate',gate_pixels)
    del z; gc.collect()

summary=pd.DataFrame(rows)
summary.to_csv(RUN/'tables'/'hitl_v89_a1_a3_by_page.csv',index=False)
summary[summary.status.str.startswith('SKIPPED',na=False)].to_csv(RUN/'tables'/'hitl_exclusions.csv',index=False)
print(summary.status.value_counts(dropna=False))


## 5. Lock outputs and build ranked review queues

This is an annotated-only perceptual experiment. The tables quantify disagreement and coverage, not truth-side correctness.


In [ ]:
usable=summary[summary.status=='PROCESSED'].copy()
require(len(usable)>0,'No processed HITL pages are available.')

queue_specs={
    'largest_a1_vs_v89':('a1_vs_v89_changed_pixels',False),
    'largest_a3_vs_v89':('a3_vs_v89_changed_pixels',False),
    'largest_a1_vs_a3':('a1_vs_a3_changed_pixels',False),
    'highest_a3_gate_fraction':('a3_gate_fraction_of_ambiguous',False),
    'lowest_a3_gate_fraction':('a3_gate_fraction_of_ambiguous',True),
    'largest_ambiguous_population':('ambiguous_pixels',False),
    'largest_authority':('authority_pixels',False),
}
queue_rows=[]
for queue,(column,ascending) in queue_specs.items():
    q=usable.sort_values(column,ascending=ascending).head(50).copy()
    q.insert(0,'queue',queue); q.insert(1,'rank',range(1,len(q)+1)); queue_rows.append(q)
queues=pd.concat(queue_rows,ignore_index=True)
queues.to_csv(RUN/'tables'/'visual_review_queues.csv',index=False)

hash_rows=[]
for p in sorted((RUN/'pages').rglob('*')):
    if p.is_file(): hash_rows.append({'path':str(p.relative_to(RUN)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
hashes=pd.DataFrame(hash_rows); hashes.to_csv(RUN/'audit'/'page_output_hashes.csv',index=False)
lock={
    'state':'CLOSED_NO_CLEAN_TRUTH_AVAILABLE',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'reviewed_roster_pages':int(len(roster)),
    'processed_pages':int(len(usable)),
    'skipped_pages':int(len(summary)-len(usable)),
    'comparators':['V89_INCUMBENT','A1_PRESERVE_ALL_AMBIGUOUS','A3_PIXEL_GATE'],
    'pprint_gate':PPRINT_GATE,
    'v89_operator_sha256':sha256_file(V89_OPERATOR_PATH),
    'output_hash_table_sha256':hashlib.sha256(hashes.to_csv(index=False).encode()).hexdigest(),
    'clean_truth_used':False,
    'promotion_authority':False,
}
(RUN/'comparison_lock.json').write_text(json.dumps(lock,indent=2))
print(json.dumps(lock,indent=2))


## 6. Whole-page and diagnostic contact sheets

Primary sheets show the actual page outputs at identical scale:

`Observed | V8.9 | A1 | A3`

Secondary sheets localise the A1-versus-A3 disagreement. Difference maps are locators, not the primary quality judgement.


In [ ]:
def load_page_image(page,name,colour=False):
    flag=cv2.IMREAD_COLOR if colour else cv2.IMREAD_GRAYSCALE
    return cv2.imread(str(RUN/'pages'/page/name),flag)

def make_whole_sheet(rows,name,max_pages=MAX_CONTACT_PAGES):
    rows=rows.head(max_pages)
    if rows.empty:return
    fig,axes=plt.subplots(len(rows),4,figsize=(20,max(6,6*len(rows))),squeeze=False)
    files=['observed.png','V89_INCUMBENT.png','A1_PRESERVE_ALL_AMBIGUOUS.png','A3_PIXEL_GATE.png']
    labels=['Observed','V8.9 incumbent','A1 preserve all AMBIGUOUS','A3 pixel gate']
    for i,(_,row) in enumerate(rows.iterrows()):
        page=str(row.page_key)
        for j,(f,label) in enumerate(zip(files,labels)):
            im=load_page_image(page,f)
            axes[i,j].imshow(im,cmap='gray',vmin=0,vmax=255)
            axes[i,j].set_title(f'{page} | {label}')
            axes[i,j].axis('off')
    plt.tight_layout(); plt.savefig(RUN/'review'/f'{name}.png',dpi=120,bbox_inches='tight'); plt.close(fig)

def tint(gray,masks):
    rgb=cv2.cvtColor(gray,cv2.COLOR_GRAY2RGB); layer=rgb.copy(); alpha=np.zeros(gray.shape,np.float32)
    for mask,colour,opacity in masks:
        layer[mask]=colour; alpha[mask]=np.maximum(alpha[mask],opacity)
    return (rgb*(1-alpha[...,None])+layer*alpha[...,None]).astype(np.uint8)

def make_diagnostic_sheet(rows,name,max_pages=MAX_CONTACT_PAGES):
    rows=rows.head(max_pages)
    if rows.empty:return
    fig,axes=plt.subplots(len(rows),5,figsize=(25,max(6,6*len(rows))),squeeze=False)
    for i,(_,row) in enumerate(rows.iterrows()):
        page=str(row.page_key); d=RUN/'pages'/page
        obs=read_gray(d/'observed.png'); a1=read_gray(d/'A1_PRESERVE_ALL_AMBIGUOUS.png'); a3=read_gray(d/'A3_PIXEL_GATE.png')
        amb=read_gray(d/'ambiguous.png')>0; gate=read_gray(d/'a3_gate.png')>0; disagreement=read_gray(d/'a1_vs_a3.png')>0
        diag=tint(obs,[(amb,(255,145,20),.30),(gate,(40,180,65),.70),(disagreement,(220,30,220),.75)])
        panels=[obs,a1,a3,disagreement.astype(np.uint8)*255,diag]
        labels=['Observed','A1','A3','A1 vs A3 difference','Orange AMBIGUOUS | green A3 preserve | magenta A1-only preserve']
        for j,(im,label) in enumerate(zip(panels,labels)):
            axes[i,j].imshow(im,cmap='gray' if im.ndim==2 else None,vmin=0 if im.ndim==2 else None,vmax=255 if im.ndim==2 else None)
            axes[i,j].set_title(f'{page} | {label}'); axes[i,j].axis('off')
    plt.tight_layout(); plt.savefig(RUN/'review'/f'{name}.png',dpi=120,bbox_inches='tight'); plt.close(fig)

for queue,(column,ascending) in queue_specs.items():
    q=usable.sort_values(column,ascending=ascending)
    make_whole_sheet(q,f'{queue}_whole_page')
    if queue in {'largest_a1_vs_v89','largest_a3_vs_v89','largest_a1_vs_a3'}:
        make_diagnostic_sheet(q,f'{queue}_diagnostic')
print('Review sheets:',RUN/'review')


## 7. Focused A1-versus-A3 disagreement crops

These crops answer the principal review question: where A1 keeps the observed byte but A3 returns to V8.9, does A3 mainly remove useful annotation residue, or does A3 reintroduce loss of printed notation?


In [ ]:
def disagreement_boxes(mask,margin=CROP_MARGIN,min_area=2):
    n,lab,stats,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8)
    h,w=mask.shape; boxes=[]
    for cid in range(1,n):
        x,y,bw,bh,area=stats[cid]
        if area<min_area:continue
        boxes.append({'component':cid,'area':int(area),'x0':max(0,x-margin),'y0':max(0,y-margin),'x1':min(w,x+bw+margin),'y1':min(h,y+bh+margin)})
    return sorted(boxes,key=lambda q:q['area'],reverse=True)

crop_rows=[]
for _,row in usable.sort_values('a1_vs_a3_changed_pixels',ascending=False).head(50).iterrows():
    page=str(row.page_key); d=RUN/'pages'/page
    obs=read_gray(d/'observed.png'); v89=read_gray(d/'V89_INCUMBENT.png'); a1=read_gray(d/'A1_PRESERVE_ALL_AMBIGUOUS.png'); a3=read_gray(d/'A3_PIXEL_GATE.png'); diff=read_gray(d/'a1_vs_a3.png')>0
    for rank,b in enumerate(disagreement_boxes(diff)[:10],1):
        x0,y0,x1,y1=b['x0'],b['y0'],b['x1'],b['y1']
        fig,axes=plt.subplots(1,4,figsize=(16,4))
        for ax,im,title in zip(axes,[obs,v89,a1,a3],['Observed','V8.9','A1','A3']):
            ax.imshow(im[y0:y1,x0:x1],cmap='gray',vmin=0,vmax=255); ax.set_title(title); ax.axis('off')
        fig.suptitle(f'{page} A1-vs-A3 disagreement #{rank} | {b["area"]} pixels')
        plt.tight_layout(); out=RUN/'review'/f'crop_{page}_{rank:02d}.png'; plt.savefig(out,dpi=170,bbox_inches='tight'); plt.close(fig)
        crop_rows.append({'page_key':page,'rank':rank,'disagreement_component_pixels':b['area'],'x0':x0,'y0':y0,'x1':x1,'y1':y1,'crop_path':str(out)})
pd.DataFrame(crop_rows).to_csv(RUN/'tables'/'a1_vs_a3_disagreement_crop_queue.csv',index=False)
print('Disagreement crops:',len(crop_rows))


## 8. Final upload package

The bundle is deliberately curated. Heavy per-page outputs remain in the run directory and are referenced by the manifests and hashes.


In [ ]:
aggregate={
    'reviewed_roster_pages':int(len(roster)),
    'processed_pages':int(len(usable)),
    'skipped_pages':int(len(summary)-len(usable)),
    'total_ambiguous_pixels':int(usable.ambiguous_pixels.sum()),
    'total_a3_gate_pixels':int(usable.a3_gate_pixels.sum()),
    'pooled_a3_gate_fraction_of_ambiguous':float(usable.a3_gate_pixels.sum()/max(usable.ambiguous_pixels.sum(),1)),
    'total_a1_vs_v89_changed_pixels':int(usable.a1_vs_v89_changed_pixels.sum()),
    'total_a3_vs_v89_changed_pixels':int(usable.a3_vs_v89_changed_pixels.sum()),
    'total_a1_vs_a3_changed_pixels':int(usable.a1_vs_a3_changed_pixels.sum()),
    'interpretation':'No clean truth is available. Counts describe candidate disagreement and review burden, not correctness.',
    'promotion_authority':False,
}
(RUN/'comparison_summary.json').write_text(json.dumps(aggregate,indent=2))
lines=[
    'NOTEBOOK 83 - HITL V8.9 VS A1 VS A3 WHOLE-PAGE COMPARISON','',
    f'Reviewed roster pages: {aggregate["reviewed_roster_pages"]}',
    f'Processed pages: {aggregate["processed_pages"]}',
    f'Skipped pages: {aggregate["skipped_pages"]}',
    f'Total AMBIGUOUS pixels: {aggregate["total_ambiguous_pixels"]:,}',
    f'Total A3-preserved pixels: {aggregate["total_a3_gate_pixels"]:,}',
    f'Pooled A3 fraction of AMBIGUOUS: {aggregate["pooled_a3_gate_fraction_of_ambiguous"]:.4%}',
    f'A1 vs V8.9 changed pixels: {aggregate["total_a1_vs_v89_changed_pixels"]:,}',
    f'A3 vs V8.9 changed pixels: {aggregate["total_a3_vs_v89_changed_pixels"]:,}',
    f'A1 vs A3 changed pixels: {aggregate["total_a1_vs_a3_changed_pixels"]:,}','',
    'Interpretation: visual/perceptual comparison only. No clean truth and no promotion authority.',
    'Primary review: Observed | V8.9 | A1 | A3.',
    'Critical queue: largest A1-vs-A3 disagreement regions.'
]
(RUN/'executive_summary.txt').write_text('\n'.join(lines))
identity={
    'created_utc':datetime.now(timezone.utc).isoformat(),'version':'notebook83-hitl-v89-a1-a3','development_only':True,
    'clean_truth_used':False,'promotion_authority':False,'pprint_gate':PPRINT_GATE,
    'v89_operator_path':str(V89_OPERATOR_PATH),'v89_operator_sha256':sha256_file(V89_OPERATOR_PATH),
    'roster_sha256':sha256_file(RUN/'tables'/'hitl_reviewed_roster.csv'),
    'summary_sha256':sha256_file(RUN/'tables'/'hitl_v89_a1_a3_by_page.csv'),
    'comparison_lock_sha256':sha256_file(RUN/'comparison_lock.json'),
    'software':{'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()},
}
(RUN/'experiment_identity.json').write_text(json.dumps(identity,indent=2,default=str))

required=[
    RUN/'executive_summary.txt',RUN/'comparison_summary.json',RUN/'comparison_lock.json',RUN/'experiment_identity.json',
    RUN/'tables'/'hitl_reviewed_roster.csv',RUN/'tables'/'hitl_v89_a1_a3_by_page.csv',RUN/'tables'/'hitl_exclusions.csv',
    RUN/'tables'/'visual_review_queues.csv',RUN/'tables'/'a1_vs_a3_disagreement_crop_queue.csv',RUN/'audit'/'page_output_hashes.csv'
]+sorted((RUN/'review').glob('*whole_page.png'))+sorted((RUN/'review').glob('*diagnostic.png'))
for p in required:
    require(p.exists(),f'Missing upload artefact: {p}')
    shutil.copy2(p,RUN/'upload'/p.name)
manifest_rows=[]
for p in sorted((RUN/'upload').iterdir()):
    if p.is_file(): manifest_rows.append({'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest_rows).to_csv(RUN/'upload'/'upload_manifest.csv',index=False)
readme='\n'.join(lines+["",f'Underlying run directory: {RUN}',"Per-page images remain under pages/<page_key>/."])
(RUN/'upload'/'README.txt').write_text(readme)
shutil.make_archive(str(RUN/'notebook83_upload_bundle'),'zip',RUN/'upload')
print('\n'.join(lines)); print('Upload bundle:',RUN/'notebook83_upload_bundle.zip')
